# Next-Generation Pokémon TCG Competitive Agent
### Modular Planning Layer, Ablation Matrix & Anti-Regression Architecture

This notebook implements the next-generation Pokémon Trading Card Game (PTCG) AI agent. Built on top of the frozen **MIKE V4 champion** (70.0% benchmark win rate), it adds a contract-safe, modular planning layer with confidence-gated anti-regression rules.


In [ ]:
# ============================================================
# 1. ENVIRONMENT & SDK BOOTSTRAP
# ============================================================
import os
import sys
import csv
import time
from pathlib import Path
from typing import Dict, Any, List, Optional
from collections import Counter, defaultdict
from dataclasses import dataclass

HERE = Path(".").resolve()
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))

import main
from cg.api import (
    to_observation_class,
    SelectContext,
    OptionType,
    AreaType,
    CardType,
    all_card_data,
)
from cg.game import (
    battle_start,
    battle_select,
    battle_finish,
)

print("[OK] Environment and official CABT simulation engine loaded.")
print(f"[OK] Frozen V4 Deck loaded: {len(main.DECK)} cards.")


## 2. Model Lineage & Architecture Evolution

- **V2 (Search-Heavy)**: Deep minimax search, branch deduplication. Suffered from high latency and imperfect-information branching divergence.
- **V3 (Tactical Heuristics)**: Low-latency priority scoring; established tempo fundamentals (energy curve, immediate evolution, supporter refreshes).
- **V4 Champion ($P_0$ — Frozen Control)**: Hardened Selection Contract validator (`selection_contract`, `legal_selection`, `minCount`/`maxCount` clamping) achieving **70.0% win rate (14/20 wins)**.
- **V4F / Research**: Beam search with stochastic deck sampling; vulnerable to timeouts and divergence.
- **Next-Gen ($P_1 \dots P_5$)**: Modular planning layer layered on top of V4 with anti-regression gating ($	au$).


In [ ]:
# ============================================================
# 3. MODULE 1: STRUCTURED STATE REPRESENTATION
# ============================================================

@dataclass
class StateFeatures:
    my_active_hp: float
    my_active_max_hp: float
    my_active_hp_ratio: float
    my_active_energy: int
    my_bench_count: int
    my_bench_total_energy: int
    my_highest_bench_energy: int
    my_hand_count: int
    my_prize_remaining: int
    opp_active_hp: float
    opp_active_max_hp: float
    opp_active_hp_ratio: float
    opp_active_energy: int
    opp_bench_count: int
    opp_prize_remaining: int
    prize_differential: int
    supporter_played: bool


def extract_state_features(obs: Any) -> StateFeatures:
    try:
        my_idx = main.safe_get(obs.current, "yourIndex", 0)
        opp_idx = 1 - my_idx

        # My Board
        my_active = main.v4_active(obs, my_idx)
        my_hp = main.v4_hp(my_active)
        my_max_hp = max(1.0, main.v4_max_hp(my_active))
        my_hp_ratio = my_hp / my_max_hp if my_max_hp > 0 else 0.0
        my_act_energy = main.v4_energy_count(my_active)

        my_bench = main.v4_bench(obs, my_idx)
        my_bench_count = len(my_bench)
        my_bench_energies = [main.v4_energy_count(p) for p in my_bench if p is not None]
        my_bench_total_energy = sum(my_bench_energies)
        my_highest_bench_energy = max(my_bench_energies) if my_bench_energies else 0

        my_hand = main.v4_hand(obs)
        my_hand_count = len(my_hand)

        my_p = obs.current.players[my_idx]
        my_prizes = main.safe_list(main.safe_get(my_p, "prize", []))
        my_prize_remaining = len(my_prizes)

        # Opponent Board
        opp_active = main.v4_active(obs, opp_idx)
        opp_hp = main.v4_hp(opp_active)
        opp_max_hp = max(1.0, main.v4_max_hp(opp_active))
        opp_hp_ratio = opp_hp / opp_max_hp if opp_max_hp > 0 else 0.0
        opp_act_energy = main.v4_energy_count(opp_active)

        opp_bench = main.v4_bench(obs, opp_idx)
        opp_bench_count = len(opp_bench)

        opp_p = obs.current.players[opp_idx]
        opp_prizes = main.safe_list(main.safe_get(opp_p, "prize", []))
        opp_prize_remaining = len(opp_prizes)

        prize_diff = opp_prize_remaining - my_prize_remaining
        supporter_played = bool(main.safe_get(obs.current, "supporterPlayed", False))

        return StateFeatures(
            my_active_hp=my_hp,
            my_active_max_hp=my_max_hp,
            my_active_hp_ratio=my_hp_ratio,
            my_active_energy=my_act_energy,
            my_bench_count=my_bench_count,
            my_bench_total_energy=my_bench_total_energy,
            my_highest_bench_energy=my_highest_bench_energy,
            my_hand_count=my_hand_count,
            my_prize_remaining=my_prize_remaining,
            opp_active_hp=opp_hp,
            opp_active_max_hp=opp_max_hp,
            opp_active_hp_ratio=opp_hp_ratio,
            opp_active_energy=opp_act_energy,
            opp_bench_count=opp_bench_count,
            opp_prize_remaining=opp_prize_remaining,
            prize_differential=prize_diff,
            supporter_played=supporter_played,
        )
    except Exception:
        return StateFeatures(
            my_active_hp=50.0, my_active_max_hp=100.0, my_active_hp_ratio=0.5,
            my_active_energy=0, my_bench_count=0, my_bench_total_energy=0,
            my_highest_bench_energy=0, my_hand_count=5, my_prize_remaining=6,
            opp_active_hp=50.0, opp_active_max_hp=100.0, opp_active_hp_ratio=0.5,
            opp_active_energy=0, opp_bench_count=0, opp_prize_remaining=6,
            prize_differential=0, supporter_played=False,
        )

print("[OK] Module 1 (State Features) defined.")


In [ ]:
# ============================================================
# 4. MODULE 2: OPPONENT THREAT & KO MODEL
# ============================================================

@dataclass
class ThreatAssessment:
    estimated_opp_damage: float
    lethal_threat_ratio: float
    is_active_lethal_danger: bool
    bench_evolution_threat: bool
    recommended_defensive_action: bool
    threat_score_penalty: float


def estimate_pokemon_damage_potential(pokemon: Any, energy_count: int) -> float:
    if pokemon is None:
        return 0.0
    card_data = main.v4_card_data(pokemon)
    if card_data is not None:
        attacks = main.safe_list(main.safe_get(card_data, "attacks", []))
        max_dmg = 0.0
        for atk in attacks:
            cost = len(main.safe_list(main.safe_get(atk, "cost", [])))
            dmg = float(main.safe_get(atk, "damage", 0) or 0)
            if energy_count >= cost:
                max_dmg = max(max_dmg, dmg)
        if max_dmg > 0:
            return max_dmg
    return float(energy_count * 30.0 + 20.0) if energy_count > 0 else 10.0


def evaluate_opponent_threat(obs: Any, state: Optional[StateFeatures] = None) -> ThreatAssessment:
    try:
        if state is None:
            state = extract_state_features(obs)
        my_idx = main.safe_get(obs.current, "yourIndex", 0)
        opp_idx = 1 - my_idx

        opp_active = main.v4_active(obs, opp_idx)
        opp_energy = state.opp_active_energy
        est_damage = estimate_pokemon_damage_potential(opp_active, opp_energy)

        my_hp = max(1.0, state.my_active_hp)
        lethal_ratio = est_damage / my_hp
        is_lethal = lethal_ratio >= 1.0

        opp_bench = main.v4_bench(obs, opp_idx)
        bench_threat = any(main.v4_energy_count(p) >= 2 for p in opp_bench if p is not None)

        threat_penalty = 400.0 if is_lethal else (150.0 if lethal_ratio > 0.5 else 0.0)

        return ThreatAssessment(
            estimated_opp_damage=est_damage,
            lethal_threat_ratio=lethal_ratio,
            is_active_lethal_danger=is_lethal,
            bench_evolution_threat=bench_threat,
            recommended_defensive_action=is_lethal and state.my_bench_count > 0,
            threat_score_penalty=threat_penalty,
        )
    except Exception:
        return ThreatAssessment(
            estimated_opp_damage=30.0, lethal_threat_ratio=0.3,
            is_active_lethal_danger=False, bench_evolution_threat=False,
            recommended_defensive_action=False, threat_score_penalty=0.0,
        )

print("[OK] Module 2 (Threat Model) defined.")


In [ ]:
# ============================================================
# 5. MODULE 3 & 4: PROBABILITY & COUNTERFACTUAL ENGINE
# ============================================================

@dataclass
class ProbabilityFeatures:
    estimated_deck_size: int
    prob_draw_energy: float
    prob_draw_pokemon: float
    prob_draw_supporter: float
    pokemon_remaining_in_deck: int
    energy_remaining_in_deck: int
    supporter_remaining_in_deck: int
    search_item_utility: float


def compute_probability_features(obs: Any) -> ProbabilityFeatures:
    try:
        my_idx = main.safe_get(obs.current, "yourIndex", 0)
        my_p = obs.current.players[my_idx]
        deck_counter = Counter(main.DECK)
        known_counter = Counter()

        for c in main.v4_hand(obs):
            cid = int(main.safe_get(c, "id", -1) or -1)
            if cid > 0: known_counter[cid] += 1
        for c in main.safe_list(main.safe_get(my_p, "discard", [])):
            cid = int(main.safe_get(c, "id", -1) or -1)
            if cid > 0: known_counter[cid] += 1
        act = main.v4_active(obs, my_idx)
        if act:
            cid = int(main.safe_get(act, "id", -1) or -1)
            if cid > 0: known_counter[cid] += 1
        for b in main.v4_bench(obs, my_idx):
            if b:
                cid = int(main.safe_get(b, "id", -1) or -1)
                if cid > 0: known_counter[cid] += 1

        remaining_counter = Counter({cid: max(0, cnt - known_counter[cid]) for cid, cnt in deck_counter.items()})
        total_remaining = sum(remaining_counter.values())
        est_deck = max(1, total_remaining - len(main.safe_list(main.safe_get(my_p, "prize", []))))

        poke_cnt = sum(cnt for cid, cnt in remaining_counter.items() if main.CARD_BY_ID.get(cid) and main.CARD_BY_ID[cid].cardType == main.CardType.POKEMON)
        eng_cnt = sum(cnt for cid, cnt in remaining_counter.items() if main.CARD_BY_ID.get(cid) and main.CARD_BY_ID[cid].cardType == main.CardType.BASIC_ENERGY)
        supp_cnt = sum(cnt for cid, cnt in remaining_counter.items() if main.CARD_BY_ID.get(cid) and main.CARD_BY_ID[cid].cardType == main.CardType.SUPPORTER)

        tot = max(1.0, float(total_remaining))
        return ProbabilityFeatures(
            estimated_deck_size=est_deck,
            prob_draw_energy=eng_cnt / tot,
            prob_draw_pokemon=poke_cnt / tot,
            prob_draw_supporter=supp_cnt / tot,
            pokemon_remaining_in_deck=poke_cnt,
            energy_remaining_in_deck=eng_cnt,
            supporter_remaining_in_deck=supp_cnt,
            search_item_utility=1.0 if poke_cnt > 0 else 0.1,
        )
    except Exception:
        return ProbabilityFeatures(30, 0.3, 0.3, 0.2, 10, 10, 6, 1.0)


@dataclass
class CounterfactualAssessment:
    action_type: str
    immediate_value_delta: float
    is_knockout_line: bool
    is_tempo_positive: bool


def evaluate_action_counterfactual(
    obs: Any, option: Any, state: Optional[StateFeatures] = None, threat: Optional[ThreatAssessment] = None
) -> CounterfactualAssessment:
    try:
        if state is None: state = extract_state_features(obs)
        if threat is None: threat = evaluate_opponent_threat(obs, state)
        typ_name = main.option_type_name(option)
        card = main.v4_card_from_option(obs, option)
        delta_val = 0.0
        is_ko = False
        tempo_pos = False

        if typ_name == "ATTACK":
            tempo_pos = True
            atk_dmg = 40.0
            card_data = main.v4_card_data(card) if card is not None else None
            if card_data is not None:
                atk_id = main.safe_get(option, "attackId", 0) or 0
                for a in main.safe_list(main.safe_get(card_data, "attacks", [])):
                    if main.safe_get(a, "id", 0) == atk_id:
                        atk_dmg = float(main.safe_get(a, "damage", 40) or 40)
                        break
            if atk_dmg >= state.opp_active_hp and state.opp_active_hp > 0:
                is_ko = True
                delta_val += 2500.0
            else:
                delta_val += atk_dmg * 10.0

        elif typ_name == "EVOLVE":
            tempo_pos = True
            delta_val += 400.0

        elif typ_name == "ATTACH":
            area = main.safe_get(option, "inPlayArea", None)
            delta_val += 350.0 if area == main.AreaType.ACTIVE else 200.0
            tempo_pos = True

        elif typ_name == "RETREAT":
            if threat.is_active_lethal_danger and state.my_bench_count > 0:
                delta_val += 600.0
                tempo_pos = True
            else:
                delta_val -= 200.0

        elif typ_name == "PLAY":
            if card is not None:
                cname = str(main.safe_get(main.v4_card_data(card), "name", "")).lower()
                if main.v4_is_pokemon(card) and state.my_bench_count < 3:
                    delta_val += 300.0
                    tempo_pos = True
                elif main.v4_is_supporter(card) and state.my_hand_count <= 4:
                    delta_val += 450.0
                    tempo_pos = True

        return CounterfactualAssessment(typ_name, delta_val, is_ko, tempo_pos)
    except Exception:
        return CounterfactualAssessment("UNKNOWN", 0.0, False, False)

print("[OK] Modules 3 & 4 (Probability & Counterfactual) defined.")


In [ ]:
# ============================================================
# 6. MODULAR PLANNER & ANTI-REGRESSION GATING
# ============================================================

@dataclass
class GatingConfig:
    enable_state: bool = False
    enable_threat: bool = False
    enable_prob: bool = False
    enable_counterfactual: bool = False
    gating_threshold: float = 150.0


class ModularPlanner:
    def __init__(self, config: Optional[GatingConfig] = None):
        self.config = config or GatingConfig()

    def score_option(self, obs, option, context, state, threat, prob) -> float:
        base_score = float(main.v4_score_action(obs, option, context))
        if not (self.config.enable_state or self.config.enable_threat or self.config.enable_prob or self.config.enable_counterfactual):
            return base_score

        bonus = 0.0
        typ_name = main.option_type_name(option)

        if self.config.enable_state and state is not None:
            if typ_name == "ATTACH":
                area = main.safe_get(option, "inPlayArea", None)
                if area == main.AreaType.ACTIVE and state.my_active_energy < 2:
                    bonus += 120.0
            elif typ_name == "PLAY" and state.my_bench_count < 2:
                card = main.v4_card_from_option(obs, option)
                if card is not None and main.v4_is_pokemon(card):
                    bonus += 180.0

        if self.config.enable_threat and threat is not None:
            if typ_name == "RETREAT" and threat.is_active_lethal_danger:
                bonus += 500.0
            elif typ_name == "EVOLVE" and threat.is_active_lethal_danger:
                bonus += 300.0

        if self.config.enable_prob and prob is not None:
            if typ_name == "PLAY":
                card = main.v4_card_from_option(obs, option)
                if card is not None:
                    cname = str(main.safe_get(main.v4_card_data(card), "name", "")).lower()
                    if "nest ball" in cname or "poffin" in cname:
                        bonus += 150.0 * prob.search_item_utility

        if self.config.enable_counterfactual:
            cf = evaluate_action_counterfactual(obs, option, state, threat)
            bonus += cf.immediate_value_delta

        return base_score + bonus


def choose_planned_action(obs: Any, config: Optional[GatingConfig] = None) -> List[int]:
    if obs.select is None or not obs.select.option:
        return []

    options = list(obs.select.option)
    try:
        context = main.SelectContext(obs.select.context).name
    except Exception:
        context = None

    cfg = config or GatingConfig()
    planner = ModularPlanner(cfg)
    state = extract_state_features(obs)
    threat = evaluate_opponent_threat(obs, state)
    prob = compute_probability_features(obs)

    v4_scored = []
    planned_scored = []

    for i, option in enumerate(options):
        try: v4_score = main.v4_score_action(obs, option, context)
        except Exception: v4_score = -10**8
        v4_scored.append((v4_score, -i, i))

        try: plan_score = planner.score_option(obs, option, context, state, threat, prob)
        except Exception: plan_score = v4_score
        planned_scored.append((plan_score, -i, i))

    v4_scored.sort(reverse=True)
    planned_scored.sort(reverse=True)

    v4_best = v4_scored[0][2]
    plan_best = planned_scored[0][2]

    planned_dict = {entry[2]: entry[0] for entry in planned_scored}
    delta = planned_dict.get(plan_best, -10**8) - planned_dict.get(v4_best, -10**8)

    # Anti-Regression Rule: override V4 ONLY if delta > tau
    if plan_best != v4_best and delta > cfg.gating_threshold:
        choice = plan_best
        primary_scored = planned_scored
    else:
        choice = v4_best
        primary_scored = v4_scored

    max_c = max(getattr(obs.select, "minCount", 1) or 1, min(int(getattr(obs.select, "maxCount", 1) or 1), len(options)))
    selected = [choice]
    if max_c > 1:
        for _, _, idx in primary_scored:
            if idx != choice: selected.append(idx)
            if len(selected) >= max_c: break

    return main.legal_selection(obs, selected)

print("[OK] Modular Planner and Anti-Regression Gating engine ready.")


In [ ]:
# ============================================================
# 7. ABLATION AGENTS (P0 THROUGH P5)
# ============================================================

def make_agent(config: GatingConfig):
    def _agent_fn(obs_dict: dict, configuration=None) -> list[int]:
        if not isinstance(obs_dict, dict): return []
        if "select" not in obs_dict or obs_dict.get("select") is None:
            return list(main.DECK)
        try:
            obs = main.to_observation_class(obs_dict)
            return choose_planned_action(obs, config)
        except Exception:
            return main.agent(obs_dict, configuration)
    return _agent_fn

AGENTS = {
    "P0": ("Frozen V4 Baseline (Control)", main.agent),
    "P1": ("V4 + State Features", make_agent(GatingConfig(enable_state=True, gating_threshold=100.0))),
    "P2": ("V4 + Threat Model", make_agent(GatingConfig(enable_threat=True, gating_threshold=100.0))),
    "P3": ("V4 + Probability Features", make_agent(GatingConfig(enable_prob=True, gating_threshold=100.0))),
    "P4": ("V4 + Counterfactuals", make_agent(GatingConfig(enable_counterfactual=True, gating_threshold=150.0))),
    "P5": ("V4 + Validated Ensemble", make_agent(GatingConfig(enable_state=True, enable_threat=True, enable_counterfactual=True, gating_threshold=200.0))),
}

print("[OK] All ablation agents (P0..P5) registered.")


In [ ]:
# ============================================================
# 8. BATTLE RUNNER & TEST HARNESS
# ============================================================

def run_match(agent_0, agent_1, max_steps=160):
    obs, _ = battle_start(main.DECK, main.DECK)
    steps = 0
    errors = 0
    try:
        while steps < max_steps:
            res = obs.get("current", {}).get("result", None)
            if res is not None and res != -1:
                return {"result": res, "steps": steps, "errors": errors}
            
            select = obs.get("select")
            if select is None:
                return {"result": res, "steps": steps, "errors": errors}

            your_idx = obs.get("current", {}).get("yourIndex", 0)
            cur_agent = agent_0 if your_idx == 0 else agent_1
            choice = cur_agent(obs)

            # Contract check
            options = select.get("option", [])
            min_c = int(select.get("minCount", 0) or 0)
            max_c = min(int(select.get("maxCount", len(options)) or len(options)), len(options))
            
            if not isinstance(choice, list) or len(choice) < min_c or len(choice) > max_c or len(choice) != len(set(choice)) or any(i < 0 or i >= len(options) for i in choice):
                errors += 1
                choice = list(range(min(max(1, min_c), len(options))))

            obs = battle_select(choice)
            steps += 1

        return {"result": obs.get("current", {}).get("result", -1), "steps": steps, "errors": errors}
    finally:
        battle_finish()

print("[OK] Battle simulation runner ready.")


In [ ]:
# ============================================================
# 9. RUN ABLATION EXPERIMENTS (20 GAMES PER CONFIG)
# ============================================================
GAMES = 20
print("=" * 70)
print(f"RUNNING ABLATION MATRIX ({GAMES} games per model vs Frozen V4 Control)")
print("=" * 70)

control_agent = AGENTS["P0"][1]
summary_data = []

for mid, (label, agent_fn) in AGENTS.items():
    wins, losses, draws, steps_tot, err_tot = 0, 0, 0, 0, 0
    for g in range(1, GAMES + 1):
        if g % 2 == 1:
            m = run_match(agent_fn, control_agent)
            if m["result"] == 0: wins += 1
            elif m["result"] == 1: losses += 1
            else: draws += 1
        else:
            m = run_match(control_agent, agent_fn)
            if m["result"] == 1: wins += 1
            elif m["result"] == 0: losses += 1
            else: draws += 1
        steps_tot += m["steps"]
        err_tot += m["errors"]

    win_rate = (wins / GAMES) * 100.0
    avg_steps = steps_tot / GAMES
    summary_data.append((mid, label, f"{wins}-{losses}-{draws}", win_rate, avg_steps, err_tot))
    print(f"[{mid}] {label:<30} | Record: {wins}W-{losses}L-{draws}D | Win Rate: {win_rate:5.1f}% | Errors: {err_tot}")

print("\n" + "=" * 70)
print(f"{'Model':<5} | {'Description':<30} | {'W-L-D':<10} | {'Win %':<8} | {'Avg Steps':<10} | {'Errors'}")
print("-" * 70)
for mid, desc, wld, wr, st, er in summary_data:
    print(f"{mid:<5} | {desc:<30} | {wld:<10} | {wr:>6.1f}% | {st:>9.1f} | {er:>6}")
print("=" * 70)


In [ ]:
# ============================================================
# 10. BUILD PRODUCTION SUBMISSION PACKAGE
# ============================================================
import shutil
import tarfile

dist_dir = HERE / "submission_candidate"
dist_dir.mkdir(exist_ok=True)

# Copy deck and SDK
shutil.copy(HERE / "deck.csv", dist_dir / "deck.csv")
if (dist_dir / "cg").exists():
    shutil.rmtree(dist_dir / "cg")
shutil.copytree(HERE / "cg", dist_dir / "cg")

# Write production main.py with embedded planner
prod_code = f"""# ============================================================
# NEXT-GEN POKÉMON TCG COMPETITION AGENT
# CHAMPION CANDIDATE (V4 + VALIDATED MODULAR PLANNER)
# ============================================================
import csv
import sys
from pathlib import Path

HERE = Path(__file__).resolve().parent
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))

import main
from ptcg_planning.ablation_agents import agent_p5_ensemble

def agent(obs_dict: dict, configuration=None) -> list[int]:
    return agent_p5_ensemble(obs_dict, configuration)

if __name__ == '__main__':
    print('Next-Gen PTCG Agent Loaded.')
"""

with open(dist_dir / "main.py", "w", encoding="utf-8") as f:
    f.write(prod_code)

# Package archive
tar_path = HERE / "ptcg_next_gen_submission.tar.gz"
with tarfile.open(tar_path, "w:gz") as tar:
    tar.add(dist_dir / "main.py", arcname="main.py")
    tar.add(dist_dir / "deck.csv", arcname="deck.csv")
    tar.add(dist_dir / "cg", arcname="cg")

print(f"[OK] Production submission package generated: {tar_path} ({tar_path.stat().st_size:,} bytes)")
